# Join the 09f cluster assignments to the two-score model output

Match on `comment_id`. Preserve the 28,230 comments, clusters, and topic fields from 09f, and add `p_support` and `p_skeptic` from the newer scoring file. This notebook does not recompute the scores or modify either input.


In [1]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

DATA_DIR = Path(r'E:\Dataminingproject\Data')
CLUSTER_READY = DATA_DIR / '09f_interpretable_assignments.parquet'
SCORED = DATA_DIR / 'reddit_climate_stance_scored.parquet'
OUTPUT = DATA_DIR / '09f_interpretable_assignments_with_two_scores.parquet'
BATCH_SIZE = 20_000

for path in (CLUSTER_READY, SCORED):
    if not path.is_file():
        raise FileNotFoundError(f'Input file not found: {path}')
    print(f'Found: {path.name} ({path.stat().st_size / 1024**2:,.1f} MB)')


Found: 09f_interpretable_assignments.parquet (10.6 MB)
Found: reddit_climate_stance_scored.parquet (1,031.4 MB)


## 1. Check columns and read only IDs and scores

The scoring file is large, so the join reads only its three required columns.


In [3]:
selected_file = pq.ParquetFile(CLUSTER_READY)
scored_file = pq.ParquetFile(SCORED)
selected_columns = selected_file.schema_arrow.names
scored_columns = scored_file.schema_arrow.names
SCORE_COLUMNS = ['p_support', 'p_skeptic']

assert 'comment_id' in selected_columns
assert not set(SCORE_COLUMNS) & set(selected_columns)
assert set(['comment_id'] + SCORE_COLUMNS) <= set(scored_columns)

scores = pd.read_parquet(SCORED, columns=['comment_id'] + SCORE_COLUMNS)
scores['comment_id'] = scores['comment_id'].astype('string')
scores = scores.dropna(subset=['comment_id'])

duplicates = scores['comment_id'].duplicated().sum()
assert duplicates == 0, f'The scoring file contains {duplicates} duplicate comment_id values'

score_by_id = scores.set_index('comment_id')
print(f'09f cluster file: {selected_file.metadata.num_rows:,} rows')
print(f'Two-score file: {scored_file.metadata.num_rows:,} rows；valid unique IDs: {len(score_by_id):,}')


09f cluster file: 28,230 rows
Two-score file: 1,894,876 rows；valid unique IDs: 1,894,876


## 2. Match in batches and write a new Parquet file

If the output already exists, generation is skipped to avoid overwriting it. Remove or rename the old output only when intentionally regenerating it.


In [4]:
if OUTPUT.exists():
    print(f'Output already exists; keeping it: {OUTPUT}')
else:
    temporary = OUTPUT.with_name(OUTPUT.name + '.partial')
    if temporary.exists():
        raise FileExistsError(f'Leftover temporary file; inspect it before removing: {temporary}')

    output_schema = selected_file.schema_arrow
    for name in SCORE_COLUMNS:
        output_schema = output_schema.append(scored_file.schema_arrow.field(name))
    read_rows = matched_rows = 0
    try:
        with pq.ParquetWriter(temporary, output_schema, compression='snappy') as writer:
            for batch in selected_file.iter_batches(batch_size=BATCH_SIZE):
                selected = batch.to_pandas()
                read_rows += len(selected)
                selected['comment_id'] = selected['comment_id'].astype('string')
                matched = selected.loc[selected['comment_id'].isin(score_by_id.index)].copy()
                if matched.empty:
                    continue

                for name in SCORE_COLUMNS:
                    matched[name] = matched['comment_id'].map(score_by_id[name])
                matched_rows += len(matched)
                writer.write_table(pa.Table.from_pandas(matched, schema=output_schema, preserve_index=False))
        temporary.replace(OUTPUT)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(f'09f input: {read_rows:,} rows')
    print(f'Matched: {matched_rows:,} rows')
    print(f'Without a new score: {read_rows - matched_rows:,} rows')
    print(f'Saved: {OUTPUT}')


Output already exists; keeping it: E:\Dataminingproject\Data\09f_interpretable_assignments_with_two_scores.parquet


## 3. Verify and preview the result

`p_support` and `p_skeptic` are two independent model scores; `primary_cluster` and the topic names come from 09f. The downstream association-rule notebook reads this output.


In [5]:
result_file = pq.ParquetFile(OUTPUT)
print(f'Output rows: {result_file.metadata.num_rows:,}')
print('Contains p_support:', 'p_support' in result_file.schema_arrow.names)
print('Contains p_skeptic:', 'p_skeptic' in result_file.schema_arrow.names)

preview_columns = [
    c for c in
    ['comment_id', 'subreddit', 'clean_text', 'p_support', 'p_skeptic']
    if c in result_file.schema_arrow.names
]
preview = next(
    result_file.iter_batches(batch_size=5, columns=preview_columns),
    None
)
preview = (
    preview.to_pandas()
    if preview is not None
    else pd.DataFrame(columns=preview_columns)
)
display(preview)

Output rows: 28,230
Contains p_support: True
Contains p_skeptic: True


,comment_id,subreddit,clean_text,p_support,p_skeptic
0,ny0gseo,Futurology,The AI would benefit from knowing what people ...,0.001255,0.000271
1,l9xdvh6,changemyview,I think bizjets are really awesome and I will ...,0.023509,0.017078
2,lkwiiin,changemyview,You think climate change fears didn't exist in...,0.040655,0.003241
3,k9zbdqr,conspiracy,You seem to be under the impression that the w...,0.558365,0.000869
4,kfgkp70,Futurology,The world has to survive to look back on anyth...,0.946300,0.015102
